# 05 — Final results: 10k suite, baselines, fairness, red team, NNPA timing

Reads `artifacts/sim/simulate.json`, `artifacts/sim/redteam.json` and `artifacts/bench/throughput.json`.
These are the only sources for numbers on the slides and in the model card.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ART = Path(os.environ.get("NYAYA_ARTIFACTS", Path.home() / "shared" / "nyaya" / "artifacts"))
pd.set_option("display.width", 200)

In [ ]:
sim = json.loads((ART / "sim" / "simulate.json").read_text())
print("suite:", sim["suite"])
print("model:", sim["model"]["model_version"], "| Model B device:", sim["model"].get("mule_device"))
m = pd.DataFrame(sim["metrics"]).T
m[["victims_left_with_zero_rate", "victim_freeze_rate", "emergency_cash_availability", "fraud_leakage_rupees",
   "total_expected_harm_rupees", "pr_auc_model_a", "pr_auc_model_b", "conformal_coverage"]].round(4)

## Headline: victims left with Rs 0

In [ ]:
h = {k: 100 * v["victims_left_with_zero_rate"] for k, v in sim["metrics"].items()}
plt.figure(figsize=(5, 3.2)); plt.bar(list(h), list(h.values()), color=["tab:orange", "tab:gray", "tab:gray"][:len(h)])
for i, v in enumerate(h.values()):
    plt.text(i, v, f"{v:.2f}%", ha="center", va="bottom")
plt.ylabel("% of displaced payments left with Rs 0"); plt.title("Victims left with Rs 0 (10k suite)")
plt.tight_layout(); plt.show()

## Fairness by synthetic income band (never a model input; target ratio <= 1.25)

In [ ]:
f = sim["fairness"]
print("freeze ratio:", f["max_min_freeze_ratio"], "| review ratio:", f["max_min_review_ratio"], "| target met:", f["target_met"])
pd.DataFrame(f["by_sender_income_band"]).T

## Red team: we attacked our own system

In [ ]:
rt = json.loads((ART / "sim" / "redteam.json").read_text())
rows = [{"variant": "base", **{k: rt["base"][k] for k in ["rows", "detection_rate", "hold_rate", "review_rate",
         "mean_p_mule", "strongest_remaining_feature"]}}]
for v in rt["variants"]:
    rows.append({"variant": v["variant"], **{k: v[k] for k in ["rows", "detection_rate", "hold_rate", "review_rate",
                 "mean_p_mule", "strongest_remaining_feature"]}, "change_vs_base": v["detection_change_vs_base"]})
pd.DataFrame(rows).round(4)

## Model B: NNPA vs CPU (median latency incl. host-device transfer)

In [ ]:
b = json.loads((ART / "bench" / "throughput.json").read_text())
print(b["model"], "| nnpa available:", b["nnpa_available"], "| max |logit diff| cpu vs nnpa:",
      b.get("max_abs_logit_diff_cpu_vs_nnpa"))
pd.DataFrame(b["results"]).pivot(index="batch", columns="device", values=["p50_ms", "decisions_per_s"])